In [ ]:
print("hello world!")

In [ ]:
import json
import torch
from datasets import Dataset
from sentence_transformers import (
    SentenceTransformer,
    SentenceTransformerTrainer,
    SentenceTransformerTrainingArguments,
)
from sentence_transformers.losses import MultipleNegativesRankingLoss
from sentence_transformers.training_args import BatchSamplers
from sentence_transformers.evaluation import TripletEvaluator

In [ ]:
# ── Load Data ──────────────────────────────────────
def load_json(path):
    with open(path, "r", encoding="utf-8") as f:
        return json.load(f)

train_data = load_json("/kaggle/input/datasets/muhammadzuamaala/datasetujicobae/train.json")
val_data   = load_json("/kaggle/input/datasets/muhammadzuamaala/datasetujicobae/val.json")
test_data  = load_json("/kaggle/input/datasets/muhammadzuamaala/datasetujicobae/test.json")

In [ ]:
# Format kolom sama persis seperti referensi (query, pos, neg)
def to_dataset(data):
    return Dataset.from_dict({
        "anchor"   : [d["anchor"]   for d in data],
        "positive" : [d["positive"] for d in data],
        "negative" : [d["negative"] for d in data],
    })

train_dataset = to_dataset(train_data)
val_dataset   = to_dataset(val_data)
test_dataset  = to_dataset(test_data)

print(f"Train : {len(train_dataset)}")
print(f"Val   : {len(val_dataset)}")
print(f"Test  : {len(test_dataset)}")

In [ ]:
# ── Load Model ─────────────────────────────────────
model_name = "intfloat/multilingual-e5-small"
model = SentenceTransformer(model_name)
print(f"\nModel loaded: {model_name}")

# ── Evaluator SEBELUM training (baseline) ──────────
evaluator_val = TripletEvaluator(
    anchors   = val_dataset["anchor"],
    positives = val_dataset["positive"],
    negatives = val_dataset["negative"],
    name      = "val",
)

print("\nBaseline SEBELUM fine-tuning:")
print("Validation:", evaluator_val(model))

In [ ]:
# ── Loss ───────────────────────────────────────────
loss = MultipleNegativesRankingLoss(model)

# ── Training Args ──────────────────────────────────
finetuned_model_name = "e5-multilingual-small-hukum"

args = SentenceTransformerTrainingArguments(
    output_dir                  = f"/kaggle/working/{finetuned_model_name}",
    num_train_epochs            = 10,
    per_device_train_batch_size = 16,
    per_device_eval_batch_size  = 16,
    learning_rate               = 2e-5,
    warmup_ratio                = 0.1,
    batch_sampler               = BatchSamplers.NO_DUPLICATES,  # ← penting untuk MNRL
    eval_strategy               = "steps",
    eval_steps                  = 10,
    logging_steps               = 10,
    fp16                        = True,
)

# ── Trainer ────────────────────────────────────────
trainer = SentenceTransformerTrainer(
    model         = model,
    args          = args,
    train_dataset = train_dataset,
    eval_dataset  = val_dataset,
    loss          = loss,
    evaluator     = evaluator_val,
)

In [ ]:
# ── Training ───────────────────────────────────────
trainer.train()


In [ ]:
# ── Evaluasi SESUDAH training ──────────────────────
evaluator_test = TripletEvaluator(
    anchors   = test_dataset["anchor"],
    positives = test_dataset["positive"],
    negatives = test_dataset["negative"],
    name      = "test",
)

print("\n===== HASIL AKHIR =====")
print("Validation :", evaluator_val(model))
print("Test       :", evaluator_test(model))

In [ ]:
from sentence_transformers import util

# Query baru yang tidak ada di dataset
test_queries = [
    "query: Apa hukuman bagi seseorang yang menyebarkan berita hoaks?",
    "query: Bagaimana jika seseorang mencuri data pribadi orang lain?",
    "query: Apakah pelaku kekerasan dalam rumah tangga bisa dipidana?",
]

# Passage dari dataset kamu (positives)
passages = [d["positive"] for d in val_data]

print("Test Generalisasi Model\n" + "="*50)
for query in test_queries:
    query_emb    = model.encode(query, convert_to_tensor=True)
    passage_embs = model.encode(passages, convert_to_tensor=True)
    
    scores = util.cos_sim(query_emb, passage_embs)[0]
    top3   = scores.topk(10)
    
    print(f"\nQuery: {query[7:]}")
    print("Top 3 hasil:")
    for score, idx in zip(top3.values, top3.indices):
        print(f"  [{score:.4f}] {passages[idx.item()][9:80]}...")
    print()

In [ ]:
# ── Save Model ─────────────────────────────────────
model.save_pretrained(f"/kaggle/working/lawe5indonesiansample/best-model")
print(f"\n✅ Model tersimpan!")

In [ ]:
from transformers import XLMRobertaTokenizer

# Override tokenizer dengan class yang universal
correct_tokenizer = XLMRobertaTokenizer.from_pretrained("intfloat/multilingual-e5-small")

# Inject ke model sebelum save
model.tokenizer = correct_tokenizer

# Baru save & push — tokenizer_config.json akan otomatis benar
save_path = "/kaggle/working/e5-multilingual-small-hukum/best-model"
model.save_pretrained(save_path)
model.push_to_hub("mzuama/newmodele5", private=False, exist_ok=True)

print("✅ Selesai! Tidak perlu manual fix lagi.")

In [ ]:
from huggingface_hub import login
login(token="xxxxxxxx") 
print("login berhasil")

In [ ]:
# Simpan ke Hugging Face Hub
model.push_to_hub(
    repo_id        = "mzuama/E5-sampel-law",  # ← ganti username
    private        = False,   # True = private, False = public
    exist_ok       = True,
)

print("✅ Model berhasil diupload ke Hugging Face!")
print("🔗 https://huggingface.co/username-kamu/e5-multilingual-small-hukum")

In [ ]:
from huggingface_hub import snapshot_download
import json, os

# Download semua file model dulu
model_path = snapshot_download(repo_id="mzuama/E5-sampel-law")
print("Model downloaded to:", model_path)

# Fix tokenizer_config.json
config_path = os.path.join(model_path, "https://huggingface.co/mzuama/E5-sampel-law/resolve/main/tokenizer_config.json")
with open(config_path) as f:
    config = json.load(f)

# Ganti class yang salah
config["tokenizer_class"] = "XLMRobertaTokenizer"  # sesuai token <s>, </s>, <mask>

with open(config_path, "w") as f:
    json.dump(config, f, indent=2)

print("Fixed! tokenizer_class sekarang:", config["tokenizer_class"])

In [ ]:
# Pastikan path-nya bener dulu
print(model_path)

# Baca ulang untuk verifikasi sudah berubah
with open(os.path.join(model_path, "tokenizer_config.json")) as f:
    print(json.dumps(json.load(f), indent=2))